
# 📶 Conectividade Brasil — Dashboard

In [ ]:
import pandas as pd
from pathlib import Path
import plotly.express as px
from ipywidgets import Output, VBox, HBox, IntSlider, SelectMultiple, Layout, Button, Label
from IPython.display import display
from PIL import Image

# -----------------------------
# Carregar dados
DATA_PATH = Path(r"D:/OneDriveBackup/OneDrive - Claro SA/LAURA SILVA SOARES DE MELO/arquivos/projetos/projeto_conectividade/data/br_anatel_indice_brasileiro_conectividade_municipio.csv")
df = pd.read_csv(DATA_PATH, sep=',', encoding='utf-8')
df.columns = [c.strip().lower() for c in df.columns]
nanos = sorted(df['ano'].dropna().unique())
ufs = sorted(df['sigla_uf'].dropna().unique())

# -----------------------------
# Variáveis globais
pri_global = pd.DataFrame()
figs_global = []

# -----------------------------
# Funções de plot

def plot_bar_ibc(dfl, ano):
    ibc_uf = dfl.groupby('sigla_uf', as_index=False)['ibc'].mean().sort_values('ibc', ascending=False)
    return px.bar(
        ibc_uf,
        x='sigla_uf',
        y='ibc',
        color='ibc',
        color_continuous_scale='Blues',
        title=f"IBC médio por UF (Ano {ano})"
    )

def plot_line_evolucao(df, ufs_sel):
    if ufs_sel:
        df_evo = df[df['sigla_uf'].isin(list(ufs_sel))].groupby(['ano','sigla_uf'], as_index=False)['ibc'].mean()
        return px.line(
            df_evo,
            x='ano',
            y='ibc',
            color='sigla_uf',
            markers=True,
            title='Evolução do IBC por UF'
        )
    else:
        df_evo = df.groupby('ano', as_index=False)['ibc'].mean()
        return px.line(
            df_evo,
            x='ano',
            y='ibc',
            markers=True,
            title='Evolução do IBC (Média Nacional)'
        )

# 👉 NOVA FUNÇÃO: Gráfico FIBRA vs IBC
def plot_scatter_fibra_ibc(dfl, ano):
    fig = px.scatter(
        dfl,
        x='fibra',
        y='ibc',
        color='sigla_uf',
        size='cobertura_pop_4g5g',
        hover_name='id_municipio',
        title=f"Relação entre Fibra (%) e IBC – Ano {ano}",
        labels={'fibra': 'Fibra (%)', 'ibc': 'IBC'}
    )
    fig.update_traces(marker=dict(opacity=0.7))
    return fig

# -----------------------------
# Área de saída
out_area = Output()

# Widgets
ano_label = Label("Ano")
ano_slider = IntSlider(min=int(min(nanos)), max=int(max(nanos)), value=int(max(nanos)), step=1, layout=Layout(width='70%'))

fibra_label = Label("Baixa Fibra (%)")
low_fibra = IntSlider(min=0, max=100, value=33, step=1, layout=Layout(width='70%'))

cob_label = Label("Alta Cobertura (%)")
high_cob = IntSlider(min=0, max=100, value=80, step=1, layout=Layout(width='70%'))

uf_label = Label("Unidades Federativas (UFs)")
uf_select = SelectMultiple(options=ufs, layout=Layout(width='70%', height='250px'))

# Botões
update_btn = Button(description="Atualizar Dashboard", button_style='primary')
export_table_btn = Button(description="Exportar Tabela Excel", button_style='success')
export_graphs_btn = Button(description="Exportar Gráficos PNG", button_style='info')
export_pdf_btn = Button(description="Exportar Gráficos PDF", button_style='warning')

# -----------------------------
# Função principal de render
def render(_=None):
    global pri_global, figs_global
    with out_area:
        out_area.clear_output()

        dfl = df[df['ano'] == ano_slider.value].copy()
        if uf_select.value:
            dfl = dfl[dfl['sigla_uf'].isin(list(uf_select.value))]

        print(f"IBC médio: {dfl['ibc'].mean():.1f}\nMunicípios: {len(dfl)}")

        # 👉 AGORA TEM 3 GRÁFICOS
        figs_global = [
            plot_bar_ibc(dfl, ano_slider.value),
            plot_line_evolucao(df, uf_select.value),
            plot_scatter_fibra_ibc(dfl, ano_slider.value)  # NOVO GRÁFICO
        ]

        for fig in figs_global:
            display(fig)

        pri_global = pd.DataFrame()
        if {'fibra','cobertura_pop_4g5g','ibc'}.issubset(dfl.columns):
            mask = (dfl['fibra'] <= low_fibra.value) & (dfl['cobertura_pop_4g5g'] >= high_cob.value)
            pri_global = dfl[mask].sort_values(['cobertura_pop_4g5g','ibc'], ascending=[False, True])
            display(pri_global.head(50))

# Vincular botão
update_btn.on_click(render)

# Layout final
controls = VBox([
    ano_label, ano_slider,
    fibra_label, low_fibra,
    cob_label, high_cob,
    uf_label, uf_select,
    HBox([update_btn, export_table_btn, export_graphs_btn, export_pdf_btn]),
    out_area
])

display(controls)

# Chamada inicial
render()
